# 02 — Veri ön işleme (preprocessing)

**Çıktılar:** `X_train`, `X_test`, `y_train`, `y_test` (ölçeklenmiş sayısal + kodlanmış kategoriler), `StandardScaler`, özellik isimleri.

**Disk:** `data/processed/train_test_split.pkl` — `03_modeling.ipynb` bu dosyayı yükler; ön işlemi tekrar yazmazsın.

**Satır:** Silme yok; **7043** satır → eğitim **5634**, test **1409** (`stratify=y`).

### İçe aktarmalar (bir sonraki hücre)
`pandas`, `numpy`, `train_test_split`, `StandardScaler`, diske yazmak için `joblib`, klasör için `Path`.

In [1]:
# --- İçe aktarmalar ve veri klasörü ---
from pathlib import Path

# Model ve tabloyu diske yazmak / okumak
import joblib
import pandas as pd
import numpy as np
# Eğitim–test bölmesi
from sklearn.model_selection import train_test_split
# Sayısal sütunları ölçeklemek (ortalama 0, std 1)
from sklearn.preprocessing import StandardScaler


def _resolve_data_dir() -> Path:
    """Jupyter çalışma dizini proje kökü veya notebooks/ olabilir; ham dosyayı bul."""
    for rel in (Path('data'), Path('../data')):
        candidate = (rel / 'WA_Fn-UseC_-Telco-Customer-Churn.xls').resolve()
        if candidate.is_file():
            return candidate.parent
    raise FileNotFoundError("WA_Fn-UseC_-Telco-Customer-Churn.xls bulunamadı. Çalışma dizinini proje kökü veya notebooks olarak ayarlayın.")


# Ham CSV ve processed çıktıları bu klasör altında
DATA = _resolve_data_dir()
# Doğru klasör seçildi mi kontrol
print("Veri klasörü:", DATA)


Veri klasörü: C:\Users\Can\Desktop\data\churn-project-yzta\data


### Ham veri önizlemesi (bir sonraki hücre, isteğe bağlı)
Boyut, `dtypes`, `head`. Asıl pipeline bir sonraki hücrede veriyi **yeniden yükler**; bu hücreyi atlayıp doğrudan oradan çalıştırabilirsin.

In [2]:
# --- İsteğe bağlı ham önizleme ---
# Ham tabloyu oku
df = pd.read_csv(DATA / "WA_Fn-UseC_-Telco-Customer-Churn.xls")

# Kaç satır/sütun
print("Ham boyut:", df.shape)
# Sütun tipleri özeti
print(df.dtypes)
# İlk satırlar
df.head()


Ham boyut: (7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Kimlik kaldırma, `TotalCharges`, hedef ayrımı (bir sonraki hücre)
- **customerID** silinir.
- **TotalCharges** sayısal; dönüşmeyenler **0** ile doldurulur.
- **Churn** → `y` (1/0); özellikler **X**. `y` için assert ile etiket bütünlüğü kontrol edilir.

In [3]:
# --- Temizlik ve hedef ayrımı ---
# Her çalıştırmada taze veri — önceki önizleme hücresine bağımlı değil
df = pd.read_csv(DATA / "WA_Fn-UseC_-Telco-Customer-Churn.xls")

# Model için anlamsız kimlik sütununu kaldır
df = df.drop(columns=["customerID"])

# Metin TotalCharges → sayı; çevrilemeyenler NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# Boş/boşluk satırlar NaN olur; çoğu tenure=0 — toplam ücret yok varsayımıyla 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Hedef: Yes→1, No→0
y = df["Churn"].map({"Yes": 1, "No": 0})
# Özellik matrisi: hedef sütunu çıkar
X = df.drop(columns=["Churn"])

# Özelliklerde eksik değer kaldı mı
print("NaN (X):", X.isnull().sum().sum())
# Tüm satırların etiketi dolu mu
assert y.isna().sum() == 0, "Churn etiketlerinde beklenmeyen değer var (Yes/No dışı)."
# Pozitif sınıf oranı
print("y dağılımı:\n", y.value_counts(normalize=True))


NaN (X): 0
y dağılımı:
 Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


### İkili kategoriler (bir sonraki hücre)
Cinsiyet ve Yes/No alanları 0/1 yapılır. Önce `astype("string").str.strip()` ile gizli boşluk riski azaltılır.

In [4]:
# --- İkili kategorileri sayıya çevir ---
# İki değerli (veya cinsiyet) sütunlar
binary_cols = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]

for col in binary_cols:
    # Nullable string + baş/son boşluk temizliği
    X[col] = X[col].astype("string").str.strip()

for col in binary_cols:
    if col == "gender":
        # Male/Female → 1/0
        X[col] = X[col].map({"Male": 1, "Female": 0})
    else:
        # Yes/No → 1/0
        X[col] = X[col].map({"Yes": 1, "No": 0})

# Map edilemeyen değer kaldıysa hata ve teşhis
if X[binary_cols].isnull().any().any():
    for col in binary_cols:
        bad = X[col].isnull().sum()
        if bad:
            print(col, "NaN sayısı:", bad, "| örnek:", df[col].dropna().unique()[:10])
    raise ValueError("Binary map sonrası NaN var.")


### One-hot, train/test, ölçekleme (bir sonraki hücre)
`get_dummies` ile çok kategorili sütunlar; `drop_first=True`. Ardından %80/%20 bölme ve `StandardScaler` (**fit** yalnızca train’de).

In [5]:
# --- One-hot, bölme, ölçekleme ---
# One-hot uygulanacak çok kategorili sütunlar
cat_cols = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
]

# Kategorileri 0/1 sütunlara aç; ilk kategoriyi düşür (çoklu doğrusal bağımlılığı azaltır)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("Özellik sayısı:", X.shape[1])

# StandardScaler ile ölçeklenecek sayısal sütunlar
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

# %80 eğitim, %20 test; churn oranı her iki parçada da benzer (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
# Ölçekleme uyarılarından kaçınmak için açık kopya
X_train = X_train.copy()
X_test = X_test.copy()
# Ortalama/std yalnızca train’den öğrenilir
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
# Test aynı dönüşümle transform edilir (fit yok)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train churn oranı:", round(float(y_train.mean()), 4))
print("Test churn oranı:", round(float(y_test.mean()), 4))
# Ölçeklenmiş train örneği
X_train.head()


Özellik sayısı: 30
X_train: (5634, 30) | X_test: (1409, 30)
Train churn oranı: 0.2654
Test churn oranı: 0.2654


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3738,1,0,0,0,0.102371,0,0,-0.521976,-0.262257,True,...,False,False,True,False,True,False,False,False,True,False
3151,1,0,1,1,-0.711743,1,0,0.337478,-0.503635,False,...,False,False,False,False,False,False,False,False,False,True
4860,1,0,1,1,-0.793155,0,0,-0.809013,-0.749883,True,...,True,False,False,False,False,False,True,False,False,True
3867,0,0,1,0,-0.263980,1,1,0.284384,-0.172722,False,...,False,False,True,False,True,False,True,True,False,False
3810,1,0,1,1,-1.281624,1,0,-0.676279,-0.989374,False,...,False,False,False,False,False,False,False,False,True,False


### Kalite kontrolü (bir sonraki hücre)
NaN, tipler, boyut ve indeks uyumu; churn oranlarının özeti.

In [6]:
# --- Doğrulama ---
# Hedef yalnızca 0 ve 1
assert set(y.unique()) <= {0, 1}
# Train’de NaN olmamalı
assert X_train.isna().sum().sum() == 0
# Test’te NaN olmamalı
assert X_test.isna().sum().sum() == 0
# object kategorisi kalmamalı (sayı veya bool)
non_numeric = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()
assert len(non_numeric) == 0, non_numeric
# Satır sayısı X ile y aynı
assert X_train.shape[0] == len(y_train)
# Aynı satır sırası (indeks eşleşmesi)
assert (X_train.index == y_train.index).all()

# Oranların özeti (stratify sonrası birbirine yakın olmalı)
print("Global churn oranı:", round(float(y.mean()), 4))
print("Train churn oranı:", round(float(y_train.mean()), 4))
print("Test  churn oranı:", round(float(y_test.mean()), 4))
print("\nTüm kontroller geçti.")


Global churn oranı: 0.2654
Train churn oranı: 0.2654
Test  churn oranı: 0.2654

Tüm kontroller geçti.


### Diske kayıt (bir sonraki hücre)
`joblib` ile `data/processed/train_test_split.pkl` oluşturulur. API veya başka ortamda aynı ölçekleyici ve sütun sırası gerekir.

In [7]:
# --- İşlenmiş train/test paketini kaydet ---
# 03_modeling.ipynb bu dosyadan yükleyecek
processed_dir = DATA / 'processed'
# Klasör yoksa oluştur
processed_dir.mkdir(parents=True, exist_ok=True)
# Tek dosyada paket
out_path = processed_dir / 'train_test_split.pkl'

# Tüm parçaları sözlükte topla
bundle = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "scaler": scaler,
    "num_cols": num_cols,
    "feature_columns": X_train.columns.tolist(),
    "random_state": 42,
    "test_size": 0.2,
}
# Diske serileştir
joblib.dump(bundle, out_path)
# Mutlak yolu yazdır
print("Kaydedildi:", out_path.resolve())


Kaydedildi: C:\Users\Can\Desktop\data\churn-project-yzta\data\processed\train_test_split.pkl
